# Libraries

In [9]:
import os

os.chdir("..")

import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

# Load data

We are going to work only with 3 variables to start

In [96]:
train_df = pd.read_csv(
    "/Users/luisfernandocorcueraleon/Desktop/code/Credit-Score-Classification/data/train.csv"
)

train_sample_df = train_df[["Occupation", "Annual_Income", "Num_Credit_Inquiries"]]
target = train_df["Credit_Score"]

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_60127/3808375836.py:1: DtypeWarning: Columns (0: Monthly_Balance) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(


In [97]:
print(train_sample_df.shape)
display(train_sample_df.head())
display(target.head())
print(target.value_counts())

(100000, 3)


,Occupation,Annual_Income,Num_Credit_Inquiries
0,Scientist,19114.12,4.00
1,Scientist,19114.12,4.00
2,Scientist,19114.12,4.00
3,Scientist,19114.12,4.00
4,Scientist,19114.12,4.00


0    Good
1    Good
2    Good
3    Good
4    Good
Name: Credit_Score, dtype: str

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64


**Objectives** 
1. Predict credit scoring
2. Classification problem
3. We'll reduce it into a binomial classification

# Check data

In [98]:
train_sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Occupation            100000 non-null  str    
 1   Annual_Income         100000 non-null  str    
 2   Num_Credit_Inquiries  98035 non-null   float64
dtypes: float64(1), str(2)
memory usage: 2.3 MB


In [99]:
train_sample_df["Occupation"].value_counts(normalize=True)

Occupation
_______         0.07
Lawyer          0.07
Architect       0.06
Engineer        0.06
Scientist       0.06
Mechanic        0.06
Accountant      0.06
Developer       0.06
Media_Manager   0.06
Teacher         0.06
Entrepreneur    0.06
Doctor          0.06
Journalist      0.06
Manager         0.06
Musician        0.06
Writer          0.06
Name: proportion, dtype: float64

In [100]:
train_sample_df["Annual_Income"].value_counts(normalize=True)

Annual_Income
17273.83     0.00
36585.12     0.00
20867.67     0.00
9141.63      0.00
17816.75     0.00
             ... 
41329.56_    0.00
10152115.0   0.00
38321.39_    0.00
16680.35_    0.00
37188.1_     0.00
Name: proportion, Length: 18940, dtype: float64

In [101]:
mask = train_sample_df["Annual_Income"].str.contains("_")
annual_income_check = train_sample_df[mask]
display(annual_income_check[mask].head())

annual_income_check["check_annual_income"] = annual_income_check[
    "Annual_Income"
].str.len() - annual_income_check["Annual_Income"].str.index("_")
annual_income_check["check_annual_income"].value_counts()

/var/folders/1f/54wv342x795c_zyzqlgyjjkw0000gn/T/ipykernel_60127/2999130946.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  display(annual_income_check[mask].head())


,Occupation,Annual_Income,Num_Credit_Inquiries
10,Teacher,34847.84_,2.00
27,Entrepreneur,30689.89_,4.00
32,Developer,35547.71_,4.00
56,Media_Manager,34081.38_,5.00
66,Doctor,114838.41_,3.00


check_annual_income
1    6980
Name: count, dtype: int64

In [102]:
train_sample_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Num_Credit_Inquiries,"98,035.00",27.75,193.18,0.00,3.00,6.00,9.00,"2,597.00"


# Clean data

### Annual Income

In [103]:
train_sample_df["Annual_Income_numeric"] = (
    train_sample_df["Annual_Income"].str.replace("_", "").astype("double")
)
train_sample_df.describe()

,Num_Credit_Inquiries,Annual_Income_numeric
count,"98,035.00","100,000.00"
mean,27.75,"176,415.70"
std,193.18,"1,429,618.05"
min,0.00,"7,005.93"
25%,3.00,"19,457.50"
50%,6.00,"37,578.61"
75%,9.00,"72,790.92"
max,"2,597.00","24,198,062.00"


### Num_Credit_Inquiries

In [ ]:
n_sum_values = train_sample_df["Num_Credit_Inquiries"].isnull().sum()
n_avg_values = train_sample_df["Num_Credit_Inquiries"].isnull().mean()
print(
    f"Number of null values in Num_Credit_Inquiries: {n_sum_values}\n"
    f"% of null values: {n_avg_values}"
)

Number of null values in Num_Credit_Inquiries: 1965
% of null values: 0.019649999999999945


In [105]:
avg_credit_inquires = train_sample_df["Num_Credit_Inquiries"].mean()
train_sample_df["Num_Credit_Inquiries"] = train_sample_df[
    "Num_Credit_Inquiries"
].fillna(avg_credit_inquires)

In [118]:
n_sum_values = train_sample_df["Num_Credit_Inquiries"].isnull().sum()
n_avg_values = train_sample_df["Num_Credit_Inquiries"].isnull().mean()
print(
    f"Number of null values in Num_Credit_Inquiries: {n_sum_values}\n"
    f"% of null values: {n_avg_values}"
)

Number of null values in Num_Credit_Inquiries: 0
% of null values: 0.0


In [107]:
train_sample_df.describe()

,Num_Credit_Inquiries,Annual_Income_numeric
count,"100,000.00","100,000.00"
mean,27.75,"176,415.70"
std,191.27,"1,429,618.05"
min,0.00,"7,005.93"
25%,3.00,"19,457.50"
50%,6.00,"37,578.61"
75%,9.00,"72,790.92"
max,"2,597.00","24,198,062.00"


### Credit Scoring

In [110]:
target.value_counts()

Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

In [117]:
target.unique()

<StringArray>
['Good', 'Standard', 'Poor']
Length: 3, dtype: str

In [121]:
target = target.replace({"Standard": "Good"})
target.value_counts(normalize=True)

Credit_Score
Good   0.71
Poor   0.29
Name: proportion, dtype: float64